# 89 — PXR Pharmacophore + Domain-Specific SMARTS Features

PXR has a large, flexible LBD (~1,150 Å³). Known agonist pharmacophore:
- Two hydrophobic regions (H1: planar aromatic, H2: alkyl/lipophilic)
- One H-bond acceptor (A1: C=O, N, O)
- MW 300–800, logP 2–7

Strategy:
1. Encode known PXR agonist scaffold SMARTS patterns
2. Compute PXR-specific physicochemical: logP², MW×logP, HBD×MW interactions
3. Compute VSA descriptors (MOE-style: SlogP_VSA, PEOE_VSA, SMR_VSA)
4. Append to combined features → LGBM

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
# Build idx_active / idx_inactive
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem

# Known PXR agonist scaffold SMARTS + privileged structural motifs
PXR_SMARTS = {
    # Stilbene / bibenzyl core (RXR/PXR cross-activators)
    "stilbene":      "c1ccc(/C=C/)cc1",
    # Carbamazepine-like (dibenzazepine)
    "dibenzazepine": "C1CC2=CC=CC=C2NC3=CC=CC=C13",
    # Hyperforin-like: bicyclic terpenoid with enol
    "polycyclic_oh": "[OH]C1=C(C(C)(C)C)CCCC1=O",
    # Pregnane/steroid scaffold
    "steroid_core":  "C1CC2CCC3CCCC4=CC(=O)CCC4(C)C3(C)C2(C)C1",
    # Rifampicin-like: naphthyl ketone
    "naphthylketone": "O=Cc1ccc2ccccc2c1",
    # Perfluoroalkyl (PFAS, potent PXR activators)
    "perfluoro":     "C(F)(F)F",
    # Phthalate ester (known PXR activator class)
    "phthalate":     "O=C(OCC)c1ccccc1C(=O)OCC",
    # Organochlorine (PXR activators like DDT metabolites)
    "organochloro":  "ClC(Cl)(Cl)",
    # Large fused aromatic (planar, fits PXR LBD)
    "acridine":      "c1ccc2nc3ccccc3cc2c1",
    # Sulfonamide (common in PXR active drugs)
    "sulfonamide":   "NS(=O)(=O)",
    # Macrolide-like: large ring ≥12
    "large_ring":    "[r12,r13,r14,r15,r16]",
}

def compute_pxr_features(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return [np.nan]*50

    feats = []
    # SMARTS pattern matches
    for name, smt in PXR_SMARTS.items():
        try:
            pat = Chem.MolFromSmarts(smt)
            feats.append(1.0 if mol.HasSubstructMatch(pat) else 0.0)
        except:
            feats.append(0.0)

    # PXR-specific physchem
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    tpsa = Descriptors.TPSA(mol)
    nrings = rdMolDescriptors.CalcNumRings(mol)
    narom = rdMolDescriptors.CalcNumAromaticRings(mol)
    rotb = rdMolDescriptors.CalcNumRotatableBonds(mol)
    nheavy = mol.GetNumHeavyAtoms()
    fsp3 = rdMolDescriptors.CalcFractionCSP3(mol)
    qed_score = Descriptors.qed(mol) if hasattr(Descriptors, 'qed') else np.nan

    # Interaction terms (capture PXR preference for lipophilic, medium-large)
    feats += [mw, logp, hbd, hba, tpsa, nrings, narom, rotb, nheavy, fsp3,
              logp**2,               # logP quadratic
              mw * logp,             # lipophilic bulk
              hbd * mw,              # H-bond donor × size
              (tpsa / mw) if mw>0 else np.nan,  # polar surface fraction
              (narom / nrings) if nrings>0 else 0.0,  # aromaticity ratio
              float(5.0 <= logp <= 7.0),  # PXR sweet spot logP
              float(300 <= mw <= 800),    # PXR sweet spot MW
              float(hbd <= 2),            # HBD-poor (fits PXR hydrophobic LBD)
    ]

    # VSA descriptors (MOE-style)
    try:
        slogp_vsa = list(Descriptors.SlogP_VSA_(mol))[:6]
        peoe_vsa  = list(Descriptors.PEOE_VSA_(mol))[:6]
        smr_vsa   = list(Descriptors.SMR_VSA_(mol))[:6]
        feats += slogp_vsa + peoe_vsa + smr_vsa
    except:
        feats += [np.nan]*18

    return feats

print("Computing PXR pharmacophore features...", flush=True)
X_pxr_tr = np.array([compute_pxr_features(s) for s in tr["smiles"]], dtype=np.float32)
X_pxr_te = np.array([compute_pxr_features(s) for s in te["smiles"]], dtype=np.float32)

# Impute
col_means = np.nanmean(X_pxr_tr, axis=0)
for j in range(X_pxr_tr.shape[1]):
    X_pxr_tr[np.isnan(X_pxr_tr[:,j]),j] = col_means[j]
    X_pxr_te[np.isnan(X_pxr_te[:,j]),j] = col_means[j]
print(f"Pharmacophore features: {X_pxr_tr.shape}")


Computing PXR pharmacophore features...


Pharmacophore features: (4139, 47)


In [5]:
X_aug_tr = np.hstack([X_tr, X_pxr_tr])
X_aug_te  = np.hstack([X_te, X_pxr_te])

oof = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.train(LGBM, lgb.Dataset(X_aug_tr[tr_idx], label=y_tr[tr_idx]),
                  valid_sets=[lgb.Dataset(X_aug_tr[va_idx], label=y_tr[va_idx])],
                  callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof[va_idx] = m.predict(X_aug_tr[va_idx])
    print(f"  fold {fold+1}  RAE={rae(y_tr[va_idx], oof[va_idx]):.4f}", flush=True)

m_res = full_metrics(y_tr, oof, cliff_pairs, "pxr_pharmacophore")
m_res_a = full_metrics(y_tr[active_mask], oof[active_mask], label="pharmacophore [active]")

m_final = lgb.train(LGBM, lgb.Dataset(X_aug_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_preds = np.clip(m_final.predict(X_aug_te), y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED/"oof_pxr_pharmacophore.npy", oof)
np.save(DATA_PROCESSED/"te_oof_pxr_pharmacophore.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"89_pxr_pharmacophore.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}  Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


  fold 1  RAE=0.4874


  fold 2  RAE=0.5668


  fold 3  RAE=0.5946


  fold 4  RAE=0.5724


  fold 5  RAE=0.6100


  [pxr_pharmacophore] RAE=0.5611 MAE=0.5105 R²=0.6053 r=0.7780 ρ=0.7354 τ=0.5412
  [pharmacophore [active]] RAE=3.5690 MAE=0.7484 R²=-8.9065 r=0.0504 ρ=0.0786 τ=0.0536


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\89_pxr_pharmacophore.csv  Test: min=2.45 med=4.95 max=6.05
